# Heft 3: Eine eigene Datei verproben

Passt eine Reihe, die ein Betreiber mitbringt, zu dem, was das System erwartet? Dieses Heft prüft eine CSV-Datei
gegen ein Signal: Einheit, Takt, Zeitzone, Wertebereich, Lücken, doppelte Zeitpunkte. Dann legt es die Datei neben
eine verwandte Reihe des Systems und sagt am Ende, ob und wie sie zu binden wäre.

Vorgelegt ist `beispiel-windmessung.csv`: Wind in km/h, Ortszeit ohne Zeitzone, Dezimalkomma, mit einem Ausreißer
und einer Lücke. Für eine eigene Datei die Angaben im ersten Codeblock ändern.

Das Heft liest die Datei und den Retrieval-Server. Es schreibt nur eine aufbereitete Kopie der Datei in sein
eigenes Verzeichnis; in den Broker und in das System schreibt es nichts.

In [ ]:
import os
import pathlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from opentsx.client import RetrievalClient

RETRIEVAL = os.environ.get("OPENTSX_RETRIEVAL", "http://opentsx-retrieval.opentsx.svc:8080")
SYSTEM = "h2-chain"
client = RetrievalClient(RETRIEVAL)

# ---- die Datei und wie sie geschrieben ist
FILE = pathlib.Path("beispiel-windmessung.csv")
SEP, DECIMAL, COMMENT = ";", ",", "#"
TIME_COLUMNS = ["Datum", "Uhrzeit"]          # eine Spalte, oder zwei, die zusammen den Zeitpunkt ergeben
TIME_FORMAT = "%d.%m.%Y %H:%M"
ZONE = "Europe/Berlin"                       # Zeitzone der Datei, wenn sie keine nennt; "UTC" wenn sie in UTC ist
VALUE = "Wind_kmh"
FILE_UNIT = "km/h"
TIME_MEANS = "start"                         # der Zeitpunkt nennt den Beginn des Intervalls ("start") oder sein Ende ("end")

# ---- das Signal, gegen das geprüft wird
SIGNAL = "park/DWD-01759/wind_speed"
print("Datei:", FILE.resolve(), "| Signal:", SIGNAL)

## 1. Die Datei lesen

Ortszeit wird in UTC umgerechnet. In der Nacht der Zeitumstellung im Herbst gibt es die Stunde von 02:00 bis 03:00
zweimal; sie wird nach der Reihenfolge der Zeilen zugeordnet. Im Frühjahr fehlt die Stunde von 02:00 bis 03:00.

In [ ]:
df = pd.read_csv(FILE, sep=SEP, decimal=DECIMAL, comment=COMMENT, dtype=str)
stamp = df[TIME_COLUMNS].astype(str).agg(" ".join, axis=1)
local = pd.to_datetime(stamp, format=TIME_FORMAT, errors="coerce")
unreadable = int(local.isna().sum())
if ZONE == "UTC":
    ts = local.dt.tz_localize("UTC")
else:
    ts = local.dt.tz_localize(ZONE, ambiguous="infer", nonexistent="NaT").dt.tz_convert("UTC")
text = df[VALUE].astype(str).str.strip()
values = pd.to_numeric(text.str.replace(",", ".", regex=False) if DECIMAL == "," else text, errors="coerce")
data = pd.Series(values.values, index=ts).dropna()
data = data[data.index.notna()]
if data.empty:
    raise ValueError("keine lesbaren Werte: SEP, DECIMAL, TIME_COLUMNS, TIME_FORMAT und VALUE prüfen")
print(f"{len(df)} Zeilen gelesen, {unreadable} Zeitpunkte unlesbar, {int(values.isna().sum())} Werte unlesbar")
print(f"von {data.index.min()} bis {data.index.max()} (UTC)")
data.head()

## 2. Was das System erwartet

In [ ]:
sig = client.signals(SYSTEM).set_index("id").loc[SIGNAL]
print(f"{SIGNAL}: {sig['meaning']}")
print(f"Einheit {sig['unit']}, plausibel {sig['min']} bis {sig['max']}, Betrieb {sig['operating_min']} bis {sig['operating_max']}, "
      f"Takt {sig['interval']}, Herkunft im System {sig['origin']}")

## 3. Die Prüfungen

Jede Prüfung hat ein Urteil: `ok`, `warnung` oder `fehler`, und einen Grund. Eine Warnung hindert das Binden nicht,
sie muss aber in der Bindung stehen.

In [ ]:
FACTOR = {("km/h", "m/s"): 1 / 3.6, ("m/s", "m/s"): 1.0, ("kn", "m/s"): 0.514444, ("MW", "kW"): 1000.0, ("kW", "kW"): 1.0,
          ("°C", "Cel"): 1.0, ("Cel", "Cel"): 1.0, ("hPa", "hPa"): 1.0, ("EUR/MWh", "EUR/MWh"): 1.0, ("ct/kWh", "EUR/MWh"): 10.0}
checks = []


def check(name, verdict, reason):
    checks.append({"Prüfung": name, "Urteil": verdict, "Grund": reason})


# Einheit
factor = FACTOR.get((FILE_UNIT, sig["unit"]))
if factor is None:
    check("Einheit", "fehler", f"keine Umrechnung von {FILE_UNIT} nach {sig['unit']} bekannt")
    converted = data
else:
    converted = data * factor
    check("Einheit", "ok" if factor == 1 else "warnung", f"{FILE_UNIT} nach {sig['unit']}, Faktor {factor:.6g}")

# Takt
step = data.index.to_series().diff().dropna().mode().iloc[0]
expected = pd.Timedelta(sig["interval"]) if isinstance(sig["interval"], str) else None
if expected is None:
    check("Takt", "warnung", f"die Datei hat {step}; das Signal nennt keinen Takt")
elif step == expected:
    check("Takt", "ok", f"{step}")
elif step > expected:
    check("Takt", "warnung", f"die Datei hat {step}, das Signal erwartet {expected}: gröber; ein Wert wird nicht feiner, wenn man ihn wiederholt")
else:
    check("Takt", "warnung", f"die Datei hat {step}, das Signal erwartet {expected}: feiner; sie wird beim Bereinigen gemittelt")

# Zeit: doppelte Zeitpunkte und die Bedeutung des Zeitpunkts
dupes = int(data.index.duplicated().sum())
check("Doppelte Zeitpunkte", "ok" if dupes == 0 else "fehler", f"{dupes} doppelt")
check("Zeitzone", "ok" if ZONE == "UTC" else "warnung", f"Ortszeit {ZONE}, in UTC umgerechnet" if ZONE != "UTC" else "UTC")
if TIME_MEANS == "end":
    converted.index = converted.index - step
    check("Zeitpunkt", "warnung", "die Datei stempelt das Ende eines Intervalls; auf den Beginn verschoben")
else:
    check("Zeitpunkt", "ok", "Beginn des Intervalls, wie im System")

# Wertebereich
outside = converted[(converted < sig["min"]) | (converted > sig["max"])] if pd.notna(sig["min"]) else converted.iloc[0:0]
check("Plausibler Bereich", "ok" if outside.empty else "warnung",
      f"{len(outside)} Werte außerhalb {sig['min']} bis {sig['max']}" + (f", zum Beispiel {outside.iloc[0]:.1f} am {outside.index[0]}" if len(outside) else ""))
inside = converted.drop(outside.index)
beyond = inside[(inside < sig["operating_min"]) | (inside > sig["operating_max"])] if pd.notna(sig["operating_min"]) else inside.iloc[0:0]
check("Betriebsbereich", "ok" if beyond.empty else "warnung", f"{len(beyond)} plausible Werte außerhalb des Betriebs {sig['operating_min']} bis {sig['operating_max']}")

# Lücken auf dem Takt der Datei
full = pd.date_range(data.index.min(), data.index.max(), freq=step)
missing = full.difference(data.index)
check("Lücken", "ok" if len(missing) == 0 else "warnung", f"{len(missing)} fehlende Zeitpunkte von {len(full)} ({100 * len(missing) / len(full):.1f} %)")

verdicts = pd.DataFrame(checks)
verdicts

## 4. Neben der Reihe des Systems

Die Datei, umgerechnet und ohne die Werte außerhalb des plausiblen Bereichs, neben der Rohreihe des Signals im
selben Zeitraum, beide auf den Takt der Datei gemittelt. Passen sie zusammen, ist die Datei, was sie zu sein
behauptet; weichen sie stark ab, misst sie etwas anderes oder an einem anderen Ort.

In [ ]:
start, end = inside.index.min(), inside.index.max() + step
ids = sig["clean_series"] if isinstance(sig["clean_series"], str) else sig["raw_series"]
system = client.frame(series=ids.split(","), start=start, end=end).set_index("ts")["value"].sort_index()
pair = pd.concat({"Datei": inside.resample(step).mean(), "System": system.resample(step).mean()}, axis=1).dropna()
if len(pair) > 2:
    corr = pair["Datei"].corr(pair["System"])
    bias = (pair["Datei"] - pair["System"]).mean()
    check("Vergleich mit dem System", "ok" if corr > 0.9 and abs(bias) < 0.1 * pair["System"].abs().mean() else "warnung",
          f"Korrelation {corr:.3f}, mittlere Abweichung {bias:+.2f} {sig['unit']} über {len(pair)} gemeinsame Zeitpunkte")
    fig, (a, b) = plt.subplots(1, 2, figsize=(14, 3.6), gridspec_kw={"width_ratios": [3, 1]})
    pair.plot(ax=a, lw=0.8)
    a.set_ylabel(sig["unit"])
    a.set_title("Datei und System")
    b.scatter(pair["System"], pair["Datei"], s=5)
    lim = [pair.min().min(), pair.max().max()]
    b.plot(lim, lim, color="grey", lw=0.8)
    b.set_xlabel("System")
    b.set_ylabel("Datei")
    plt.tight_layout()
    plt.show()
else:
    check("Vergleich mit dem System", "warnung", "keine gemeinsamen Zeitpunkte mit der Reihe des Systems")
pd.DataFrame(checks).tail(1)

## 5. Das Urteil und die Bindung

Ein Fehler verhindert das Binden. Warnungen gehören in die Bindung, damit jeder sieht, was die Reihe ist und was
nicht. Die aufbereitete Kopie liegt in UTC, in der Einheit des Signals und ohne die Werte außerhalb des plausiblen
Bereichs im Verzeichnis dieses Hefts; die Befehle darunter bringen sie ins Archiv und in das Topic der Domäne.
Ausgeführt werden sie von jemandem, der das darf, nicht von diesem Heft.

In [ ]:
verdicts = pd.DataFrame(checks)
errors = verdicts[verdicts["Urteil"] == "fehler"]
warnings = verdicts[verdicts["Urteil"] == "warnung"]
print("Urteil:", "nicht bindbar" if len(errors) else ("bindbar, mit Warnungen" if len(warnings) else "bindbar"))
for _, r in pd.concat([errors, warnings]).iterrows():
    print(f"  {r['Urteil']}: {r['Prüfung']}: {r['Grund']}")

OUT = pathlib.Path(f"verprobt-{SIGNAL.replace('/', '_')}.csv")
pd.DataFrame({"start": inside.index.strftime("%Y-%m-%dT%H:%M:%SZ"), "value": inside.round(4).values}).to_csv(OUT, index=False)
print(f"\naufbereitet: {OUT.resolve()} ({len(inside)} Werte)")

name = SIGNAL.replace("/", "-").lower()
metric = "partner." + SIGNAL.replace("/", ".").lower()
def iso(d):
    s = int(pd.Timedelta(d).total_seconds())
    return "PT" + (f"{s // 3600}H" if s % 3600 == 0 else f"{s // 60}M" if s % 60 == 0 else f"{s}S")


print(f"""
Bindung, wenn das Urteil es zulässt:

  bin/opentsx market import --file {OUT.name} --name {name} --metric {metric} --unit {sig['unit']} \\
      --origin MEASURED --resolution {iso(step)} --archive <archiv>
  bin/opentsx market publish --archive <archiv> --series partner/{name} --domain wind --kafscale <broker> --dry-run

und im Systemdokument (oder in der Arbeitskopie über die Seite Signals):

  raw:
    source:
      kind: topic
      topic: tsx.wind.episodes.v2
      origin: MEASURED
      select: {{ metric: {metric} }}
    note: Aus einer Datei des Betreibers, verprobt am {pd.Timestamp.now(tz='UTC'):%Y-%m-%d}. {'; '.join(warnings['Grund'])}
""")